# 03. Real benchmark: InHARD-DT -> InHARD

**Protocol.** InHARD's 16 subjects are split into 4 subject-disjoint folds (`data/processed_v2/splits.json`). For each fold, the 12 other subjects form the target training pool (unlabeled for UDA, labeled for the oracle) and the 4 fold subjects are the test set, never seen during training in any form. The source is all of InHARD-DT. Every run trains for a fixed length and is evaluated once at the end: **no target labels are used for model selection**.

**Methods.** See the docstring of `picda/experiments.py`: source-only, reverse (real->DT), DT in-domain, oracle, source+target, DANN, CORAL, MMD, PI-CDA (MMD + topology consistency), FixMatch, and few-shot (k = 1, 5, 10) from scratch / source-only / PI-CDA.

**Backbone.** Faithful CTR-GCN (Chen et al., ICCV 2021) on 19 body joints including the wrists and knuckles, 64 frames, joint coordinates.

Each run writes one JSON with its raw predictions under `output_v2/<run>/results/`. Re-running a cell **resumes**: finished runs are skipped. You can also run the same thing from a terminal with `scripts/run_benchmark.py` (better for overnight runs).

**What to send back:** the smoke-test output (or any traceback), then later the tables of notebook 04.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import torch
from picda import config as C
from picda.experiments import ALL_METHODS, Benchmark
from picda.train import TrainConfig
from picda.analysis import load_results, summary_table

print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

torch 2.10.0+cu128 | CUDA: True NVIDIA GeForce RTX 3060 Laptop GPU


## 1. Smoke test (run this first)

Every method for 1 epoch on fold 0, seed 0, into `output_v2/smoke`. The numbers are meaningless; the point is that every code path runs end to end and the per-run timings let us size the full benchmark. Expect a few minutes on a GPU.

In [2]:
smoke_cfg = TrainConfig(epochs=1, adapt_epochs=1, fewshot_steps=5, warmup_epochs=0, seeds=(0,), log_every=20)
smoke = Benchmark(smoke_cfg, C.OUTPUT_DIR / 'smoke')
smoke.run(ALL_METHODS, folds=[0], seeds=[0])
print(summary_table(load_results(C.OUTPUT_DIR / 'smoke')).to_string())

Loaded 4696 DT clips (12 subjects) and 4803 InHARD clips (16 subjects) in 4s

================ seed 0 ================


/home/g0amer/Desktop/thesis/04_Cross_Modal_Digital_Twin/picda/train.py:128: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  self.sched.step()


  [source_only s0] step 20/73 | loss=4.6332 | train_acc=0.2289 | 11s
  [source_only s0] step 40/73 | loss=2.0607 | train_acc=0.3430 | 14s
  [source_only s0] step 60/73 | loss=1.6509 | train_acc=0.4578 | 17s
  [source_only s0] step 73/73 | loss=1.5821 | train_acc=0.4423 | 19s
  -> source_only fold=all seed=0: acc=0.3779 macroF1=0.2561 (n=4803)
  -> source_only fold=0 seed=0: acc=0.3664 macroF1=0.2628 (n=1321)


/home/g0amer/Desktop/thesis/04_Cross_Modal_Digital_Twin/picda/train.py:128: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  self.sched.step()


  [reverse s0] step 20/75 | loss=4.9416 | train_acc=0.2234 | 3s
  [reverse s0] step 40/75 | loss=1.9734 | train_acc=0.3945 | 6s
  [reverse s0] step 60/75 | loss=1.6000 | train_acc=0.4742 | 9s
  [reverse s0] step 75/75 | loss=1.5305 | train_acc=0.4979 | 11s
  -> reverse_zero_shot fold=all seed=0: acc=0.2340 macroF1=0.1617 (n=4696)


/home/g0amer/Desktop/thesis/04_Cross_Modal_Digital_Twin/picda/train.py:128: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  self.sched.step()


... (15 outputs left out)

/home/g0amer/Desktop/thesis/04_Cross_Modal_Digital_Twin/picda/train.py:128: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  self.sched.step()


  [fewshot_k5_scratch f0 s0] step 5/5 | loss=6.5785 | train_acc=0.0969 | 1s
  -> fewshot_k5_scratch fold=0 seed=0: acc=0.0500 macroF1=0.0330 (n=1321)
  [fewshot_k5_source_only f0 s0] step 5/5 | loss=2.2941 | train_acc=0.3563 | 1s
  -> fewshot_k5_source_only fold=0 seed=0: acc=0.3323 macroF1=0.2701 (n=1321)
  [fewshot_k5_picda f0 s0] step 5/5 | loss=2.2448 | train_acc=0.2594 | 1s
  -> fewshot_k5_picda fold=0 seed=0: acc=0.2763 macroF1=0.2398 (n=1321)


/home/g0amer/Desktop/thesis/04_Cross_Modal_Digital_Twin/picda/train.py:128: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  self.sched.step()


  [fewshot_k10_scratch f0 s0] step 5/5 | loss=5.8029 | train_acc=0.1094 | 1s
  -> fewshot_k10_scratch fold=0 seed=0: acc=0.0167 macroF1=0.0025 (n=1321)
  [fewshot_k10_source_only f0 s0] step 5/5 | loss=2.3152 | train_acc=0.3094 | 1s
  -> fewshot_k10_source_only fold=0 seed=0: acc=0.3603 macroF1=0.2902 (n=1321)
  [fewshot_k10_picda f0 s0] step 5/5 | loss=2.3036 | train_acc=0.2938 | 1s
  -> fewshot_k10_picda fold=0 seed=0: acc=0.2604 macroF1=0.2327 (n=1321)

All requested runs finished in 4.8 min. Results in /home/g0amer/Desktop/thesis/04_Cross_Modal_Digital_Twin/output_v2/smoke
                         runs  acc_mean  acc_std  f1_mean  f1_std  bacc_mean  minutes_per_run
method                                                                                       
dt_in_domain                1     40.15      0.0    29.80     0.0      32.51             0.17
reverse_zero_shot           1     23.40      0.0    16.17     0.0      20.31             0.25
source_only                 1     36.64 

## 2. Full benchmark, seed 0 first

From the smoke-test timings on the RTX 3060 Laptop: about 0.15 s per supervised step, 0.3 s per UDA step and 0.43 s per FixMatch step. One seed (4 folds, all methods, 50 epochs) is about **8 to 9 hours**, so three seeds take about 26 hours. Run seed 0 overnight, look at it in notebook 04, then decide whether seeds 1 and 2 are worth it for every method.

The terminal route is safer for a run this long (it survives a closed VS Code window):

```
cd ~/Desktop/thesis/04_Cross_Modal_Digital_Twin
source ../03_Incremental_Learning_FLAIR/.venv/bin/activate
nohup python scripts/run_benchmark.py --seeds 0 > bench_seed0.log 2>&1 &
```

The cell below does the same thing inside the notebook. Both are resumable and write to the same folder.

In [ ]:
cfg = TrainConfig()  # defaults documented in picda/train.py
bench = Benchmark(cfg, C.OUTPUT_DIR / 'benchmark_ctrgcn')
bench.run(ALL_METHODS, seeds=[0])          # later: seeds=[1, 2]
print(summary_table(load_results(C.OUTPUT_DIR / 'benchmark_ctrgcn')).to_string())

## 3. Ablations (optional, after the main run)

* `noyaw`: no heading canonicalisation, shows how much of the gap is pure workstation orientation;
* `stgcn`: backbone sensitivity;
* `bone`: joint + bone input channels.

Each only runs the methods needed to answer its question, with seed 0 to limit cost.

In [ ]:
RUN_ABLATIONS = False
if RUN_ABLATIONS:
    core = ['source_only', 'oracle', 'mmd', 'picda']
    for name, cfg_ab in [
        ('ablation_noyaw', TrainConfig(canon_yaw='none', seeds=(0,))),
        ('ablation_stgcn', TrainConfig(model='stgcn', seeds=(0,))),
        ('ablation_bone', TrainConfig(features='joint_bone', seeds=(0,))),
    ]:
        methods = core if name != 'ablation_stgcn' else ['source_only', 'oracle', 'mmd']  # PI-CDA needs CTR-GCN topologies
        Benchmark(cfg_ab, C.OUTPUT_DIR / name).run(methods)
        print(name)
        print(summary_table(load_results(C.OUTPUT_DIR / name)).to_string())